# Pairs Trading B3 — Modelo Kalman Filter

**Documentação técnica do `05_modelo.py`**

---

## Visão geral

Este notebook documenta o pipeline completo do modelo de pairs trading com **Filtro de Kalman** para beta dinâmico.

O modelo substitui o OLS estático (uma estimativa de β por janela) por um filtro adaptativo que atualiza α e β a cada pregão.

**Referências:**
- Gatev, Goetzmann & Rouwenhorst (2006) — distância mínima
- Engle & Granger (1987) — cointegração
- Avellaneda & Lee (2010) — Kalman Filter em pairs trading

## 1. Pipeline completo

```
CSV Economatica (6M linhas)
       │
       ▼
Limpeza: remove NaN, ffill(3d), converte float64
Setores: merge com economatica_B3_setores.csv
       │
       ▼  (a cada mês — janela rolante de 504 pregões)
Liquidez: cobertura ≥80%, volume>0, q_negs>0, remove Q25
       │
       ▼
Distância mínima: D(A,B) = Σ(norm_A − norm_B)²  →  top 500 pares
       │
       ▼
Cointegração Engle-Granger: ADF nos resíduos OLS  →  p < 0,05
       │  (top 20 por p-value + distância tie-break)
       ▼
Kalman Filter: β dinâmico a cada pregão
       │
       ▼
Z-score = ν_t / √S_t  →  sinais LONG/SHORT
       │
       ▼
CSVs + Gráficos  →  output/
```

## 2. Fundamentos matemáticos

### 2.1 Por que beta dinâmico?

No OLS estático, o hedge ratio β é estimado uma vez por janela e permanece fixo. Problemas:

- **Deriva estrutural**: relações entre ativos mudam com ciclos econômicos
- **Look-ahead bias**: μ e σ do z-score calculados sobre toda a janela incluindo dados futuros
- **Latência**: o β do modelo está sempre "atrasado" em relação ao mercado

### 2.2 Formulação state-space

**Equação de observação:**
$$\log A_t = \alpha_t + \beta_t \cdot \log B_t + \varepsilon_t, \quad \varepsilon_t \sim \mathcal{N}(0, R)$$

**Equação de transição** (random walk):
$$\theta_t = \theta_{t-1} + \delta_t, \quad \delta_t \sim \mathcal{N}(0, Q)$$

onde $\theta_t = [\alpha_t, \beta_t]^\top$ e $Q = \Delta \cdot I$.

### 2.3 Equações do filtro

**Predict:**
$$\hat{\theta}_{t|t-1} = \hat{\theta}_{t-1|t-1}$$
$$P_{t|t-1} = P_{t-1|t-1} + Q$$

**Inovação:**
$$\nu_t = \log A_t - H_t \hat{\theta}_{t|t-1}, \quad H_t = [1, \log B_t]$$
$$S_t = H_t P_{t|t-1} H_t^\top + R$$

**Update:**
$$K_t = P_{t|t-1} H_t^\top / S_t \quad \text{(Kalman gain)}$$
$$\hat{\theta}_{t|t} = \hat{\theta}_{t|t-1} + K_t \nu_t$$
$$P_{t|t} = (I - K_t H_t) P_{t|t-1}$$

### 2.4 Z-score Kalman

$$Z_t = \frac{\nu_t}{\sqrt{S_t}}$$

Em regime estacionário, $Z_t \sim \mathcal{N}(0, 1)$ por construção — **sem look-ahead**.

### 2.5 O parâmetro Δ (delta)

Controla a velocidade de adaptação do β:

| Δ | Comportamento |
|---|---|
| `1e-6` | β quase fixo — par muito estável (ON/PN) |
| `1e-5` | Adaptação lenta (padrão) |
| `1e-4` | Adaptação moderada — pares macro-sensíveis |
| `1e-3` | Adaptação rápida — pares táticos |

## 3. Implementação do Kalman Filter

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#f8f9fa',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.color': '#dde1e7',
    'font.size': 10,
})

print('Bibliotecas carregadas.')

In [ ]:
def kalman_spread(log_a, log_b, delta=1e-5, ve=None, n_warm=50):
    """
    Kalman Filter para beta dinâmico em pairs trading.
    (Avellaneda & Lee, 2010)

    Parâmetros
    ----------
    log_a, log_b : arrays de log-preços
    delta        : velocidade de adaptação (Q = delta * I)
    ve           : variância do ruído de observação R
                   (None = estimada do warm-start OLS)
    n_warm       : observações para warm-start via OLS

    Retorna
    -------
    alphas : alpha dinâmico a cada t
    betas  : beta dinâmico a cada t
    z_arr  : z-score = inovação / sqrt(variância da inovação)
    innov  : inovações brutas (spread)
    S_arr  : variância da inovação
    """
    T = len(log_a)
    Q = delta * np.eye(2)

    # warm-start: OLS nos primeiros n_warm obs
    n_w = max(2, min(n_warm, T // 4))
    xw, yw = log_b[:n_w], log_a[:n_w]
    sb, sa   = xw.sum(), yw.sum()
    sbb, sab = (xw * xw).sum(), (yw * xw).sum()
    den = n_w * sbb - sb * sb

    if abs(den) > 1e-12 and n_w >= 5:
        beta0  = (n_w * sab - sb * sa) / den
        alpha0 = (sa - beta0 * sb) / n_w
        resid  = yw - alpha0 - beta0 * xw
        R = max(resid.var(), 1e-8) if ve is None else max(ve, 1e-8)
    else:
        alpha0, beta0, R = 0.0, 1.0, 0.001

    theta = np.array([alpha0, beta0], dtype=np.float64)
    P     = np.eye(2) * max(R, 1e-4)

    alphas = np.empty(T); betas = np.empty(T)
    z_arr  = np.empty(T); innov = np.empty(T); S_arr = np.empty(T)

    for t in range(T):
        H      = np.array([1.0, log_b[t]])
        P_pred = P + Q
        nu     = log_a[t] - (H @ theta)
        S      = float(H @ P_pred @ H) + R
        K      = (P_pred @ H) / S
        theta  = theta + K * nu
        P      = (np.eye(2) - np.outer(K, H)) @ P_pred
        alphas[t] = theta[0]; betas[t] = theta[1]
        innov[t]  = nu;       S_arr[t] = S
        z_arr[t]  = nu / max(S ** 0.5, 1e-10)

    return alphas, betas, z_arr, innov, S_arr

print('Função kalman_spread definida.')

## 4. Demonstração com dados sintéticos

Simulamos dois ativos cointegrados com beta que deriva ao longo do tempo.

In [ ]:
np.random.seed(42)
T = 504   # ≈ 2 anos

# beta real que deriva de 1.0 para 1.3 ao longo do período
beta_real = np.linspace(1.0, 1.3, T)
alpha_real = 0.5

# série B: random walk
log_b = np.cumsum(np.random.normal(0, 0.01, T))

# série A: cointegrada com B via beta_real variável
log_a = alpha_real + beta_real * log_b + np.random.normal(0, 0.02, T)

# aplica Kalman com delta=1e-5 (lento) e delta=1e-4 (rápido)
al_slow, be_slow, z_slow, _, _ = kalman_spread(log_a, log_b, delta=1e-5)
al_fast, be_fast, z_fast, _, _ = kalman_spread(log_a, log_b, delta=1e-4)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Demonstração Kalman Filter — beta dinâmico (dados sintéticos)',
             fontweight='bold')

# beta real vs estimado
axes[0,0].plot(beta_real, color='black', lw=1.2, label='β real')
axes[0,0].plot(be_slow, color='#2e75b6', lw=1.0, label=f'Kalman δ=1e-5 (lento)')
axes[0,0].plot(be_fast, color='#c0392b', lw=1.0, ls='--', label=f'Kalman δ=1e-4 (rápido)')
axes[0,0].set_title('β dinâmico vs real'); axes[0,0].legend(fontsize=8)
axes[0,0].set_xlabel('Pregão')

# z-scores
axes[0,1].plot(z_slow, color='#2e75b6', lw=0.8, alpha=0.8, label='δ=1e-5')
axes[0,1].plot(z_fast, color='#c0392b', lw=0.8, alpha=0.8, ls='--', label='δ=1e-4')
for lvl, col in [(2,'#c0392b'),(-2,'#1e7e34'),(0,'grey')]:
    axes[0,1].axhline(lvl, linestyle='--', color=col, lw=0.8, alpha=0.6)
axes[0,1].set_title('Z-Score Kalman'); axes[0,1].legend(fontsize=8)
axes[0,1].set_ylim(-5, 5)

# histograma dos z-scores
axes[1,0].hist(z_slow[50:], bins=30, color='#2e75b6',
               edgecolor='white', alpha=0.7, label='δ=1e-5')
axes[1,0].hist(z_fast[50:], bins=30, color='#c0392b',
               edgecolor='white', alpha=0.5, label='δ=1e-4')
from scipy.stats import norm as sp_norm
xx = np.linspace(-4, 4, 100)
n_obs = len(z_slow[50:])
axes[1,0].plot(xx, sp_norm.pdf(xx) * n_obs * (z_slow[50:].max()-z_slow[50:].min())/30,
               'k--', lw=1.5, label='N(0,1) teórica')
axes[1,0].set_title('Distribuição Z-Scores (após warm-start)')
axes[1,0].legend(fontsize=8)

# convergência de P
_, _, _, _, S_slow = kalman_spread(log_a, log_b, delta=1e-5)
axes[1,1].plot(np.sqrt(S_slow), color='#8e44ad', lw=1.0, label='√S_t (δ=1e-5)')
axes[1,1].set_title('Raiz da variância da inovação √S_t')
axes[1,1].set_xlabel('Pregão')
axes[1,1].axvline(50, linestyle='--', color='orange', lw=1.0, label='Fim warm-start')
axes[1,1].legend(fontsize=8)

plt.tight_layout()
plt.show()

from scipy.stats import skew, kurtosis
print(f'\nEstatísticas z-score (após warm-start):')
print(f'  δ=1e-5 → média={z_slow[50:].mean():.3f}  std={z_slow[50:].std():.3f}'
      f'  skew={skew(z_slow[50:]):.3f}  kurt={kurtosis(z_slow[50:]):.3f}')
print(f'  δ=1e-4 → média={z_fast[50:].mean():.3f}  std={z_fast[50:].std():.3f}'
      f'  skew={skew(z_fast[50:]):.3f}  kurt={kurtosis(z_fast[50:]):.3f}')

## 5. Seleção de liquidez (notebook 03)

Para cada janela de 504 pregões, o universo de ativos é filtrado por:

| Critério | Valor |
|---|---|
| Cobertura mínima de pregões | ≥ 80% |
| Volume financeiro mediano | > 0 |
| Quantidade de negócios mediana | > 0 |
| Remoção dos menos líquidos | Bottom 25% de volume E q_negs |

**Por que não top-100 fixo?**  
A seleção dinâmica por liquidez captura IPOs, delisting e mudanças de liquidez ao longo do tempo. Um universo fixo de 2010 incluiria ativos que perdem liquidez ou saem da bolsa.

In [ ]:
def selecionar_liquidez_demo(df_janela, cobertura_min=0.80):
    """
    Replica a lógica do notebook 03 e do 05_modelo.py.
    
    df_janela: DataFrame com colunas Ativo, Data, Volume_BRL_k, Q_Negs
    """
    tot_preg = df_janela['Data'].nunique()
    
    stats = df_janela.groupby('Ativo').agg(
        Dias     =('Data',        'nunique'),
        Vol_Med  =('Volume_BRL_k','median'),
        Negs_Med =('Q_Negs',     'median'),
    ).reset_index()
    
    # filtros básicos
    stats = stats[
        (stats['Dias'] / tot_preg >= cobertura_min) &
        (stats['Vol_Med']  > 0) &
        (stats['Negs_Med'] > 0)
    ]
    
    # remove bottom 25%
    if len(stats) >= 4:
        stats = stats[
            (stats['Vol_Med']  >= stats['Vol_Med'].quantile(0.25)) &
            (stats['Negs_Med'] >= stats['Negs_Med'].quantile(0.25))
        ]
    
    return stats['Ativo'].tolist()

print('Função selecionar_liquidez definida.')
print('Esta função é idêntica à usada no notebook 03 e no 05_modelo.py.')

## 6. Distância mínima (Gatev et al., 2006)

### Por que distância e não correlação?

A correlação mede apenas **direção linear** — dois ativos podem ter alta correlação mas divergir estruturalmente de nível. A distância euclidiana sobre preços normalizados captura tanto direção quanto nível:

$$D(A, B) = \sum_{t=1}^{T} \left(\frac{P_A^t}{P_A^0} - \frac{P_B^t}{P_B^0}\right)^2$$

### Normalização por-coluna (notebook 04)

Cada série é dividida pelo seu **primeiro preço válido independente**, não pelo preço da mesma data. Isso respeita que ativos diferentes entram no universo em datas diferentes.

### OBS_MINIMAS_PAR = 403

O check de mínimo de observações é feito **por par** (não por janela inteira). Pares com dados insuficientes são descartados individualmente — a janela continua processando outros pares.

In [ ]:
def distancia_candidatos_demo(precos_raw, cols, obs_min=403, max_cand=500):
    """
    Replica a lógica do notebook 04 e do 05_modelo.py.
    
    precos_raw : DataFrame com ffill, sem dropna global
    cols       : lista de tickers a considerar
    """
    precos_dist = precos_raw[cols]
    
    # normalização por 1º preço válido de cada série
    primeiro = precos_dist.apply(
        lambda c: c.dropna().iloc[0] if c.dropna().shape[0] > 0 else np.nan
    )
    norm_arr = (precos_dist / primeiro).values.astype(np.float64)
    
    N = len(cols)
    i_idx, j_idx = np.triu_indices(N, k=1)
    dists = np.full(len(i_idx), np.inf)
    
    for k, (ii, jj) in enumerate(zip(i_idx, j_idx)):
        ca, cb = norm_arr[:, ii], norm_arr[:, jj]
        mask   = ~(np.isnan(ca) | np.isnan(cb))
        # check OBS_MINIMAS_PAR por par (notebook 04)
        if mask.sum() < obs_min:
            continue
        dists[k] = np.sum((ca[mask] - cb[mask]) ** 2)
    
    validos = np.where(np.isfinite(dists))[0]
    n_cand  = min(max_cand, len(validos))
    if n_cand == 0:
        return []
    
    top_rel = np.argpartition(dists[validos], n_cand - 1)[:n_cand]
    top_rel = top_rel[np.argsort(dists[validos][top_rel])]
    top_pos = validos[top_rel]
    
    return [(cols[i_idx[p]], cols[j_idx[p]], float(dists[p])) for p in top_pos]

print('Função distancia_candidatos definida.')

## 7. Cointegração Engle-Granger

Para cada par que passou no filtro de distância:

$$\log P_A^t = \alpha + \beta \cdot \log P_B^t + \varepsilon_t$$

Aplica-se o **ADF** no resíduo $\varepsilon_t$:

- $H_0$: $\varepsilon_t$ tem raiz unitária → par **não** cointegrado  
- $H_1$: $\varepsilon_t$ é estacionário → par cointegrado

**Critério**: $p\text{-value} < 0{,}05$

### Papel da cointegração com Kalman

O Kalman não precisa da cointegração para funcionar matematicamente — ele filtra qualquer par. O teste serve como **filtro de qualidade**: valida que existe uma relação de equilíbrio de longo prazo antes de usar Kalman para rastrear esse equilíbrio adaptivamente.

### RAPIDO = False (padrão)

Por padrão o modelo usa `autolag='aic'` (como no notebook 04) para seleção automática de defasagens. `RAPIDO = True` fixa `maxlag=5`, reduzindo ~20% do tempo mas podendo alterar p-values marginais.

## 8. Z-score e sinais operacionais

### Diferença OLS vs Kalman

| | OLS Estático | Kalman Dinâmico |
|---|---|---|
| β | Fixo na janela | Atualizado a cada pregão |
| Spread | $\log A - \hat{\alpha} - \hat{\beta} \log B$ | Inovação $\nu_t$ |
| Z-score | $(\text{spread} - \mu) / \sigma$ | $\nu_t / \sqrt{S_t}$ |
| Look-ahead | Sim (μ, σ usam janela completa) | Não (baseado em previsão t-1) |
| Distribuição | Aprox. N(0,1) | N(0,1) por construção |

### Regras de trading

| Condição | Direção | Ação |
|---|---|---|
| $Z_t \geq +2$ | SHORT | Vende A, compra $\beta_t$ unidades de B |
| $Z_t \leq -2$ | LONG  | Compra A, vende $\beta_t$ unidades de B |
| $|Z_t| \leq 0{,}5$ | — | Fecha posição (convergência) |
| $|Z_t| \geq 3$ | — | Stop-loss (divergência) |
| $t > 30$ pregões | — | Stop por tempo |

O $\beta_t$ no momento da entrada define o hedge ratio da operação. Se $\beta_t = 1{,}2$, para cada 1 unidade de A vendida, compra-se 1,2 unidades de B.

In [ ]:
# Demonstração visual: z-score Kalman com sinais de trading
np.random.seed(7)
T = 504
beta_real2 = 1.0 + 0.2 * np.sin(np.linspace(0, 2*np.pi, T))   # beta oscilante
log_b2 = np.cumsum(np.random.normal(0, 0.01, T))
log_a2 = 0.3 + beta_real2 * log_b2 + np.random.normal(0, 0.025, T)

_, betas2, z2, _, _ = kalman_spread(log_a2, log_b2, delta=1e-5)

# identifica sinais
entradas_long  = np.where(np.diff((z2 <= -2).astype(int)) == 1)[0] + 1
entradas_short = np.where(np.diff((z2 >=  2).astype(int)) == 1)[0] + 1

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('Z-Score Kalman com Sinais de Trading e Beta Dinâmico',
             fontweight='bold')

# z-score
t_arr = np.arange(T)
axes[0].plot(t_arr, z2, color='#2e75b6', lw=0.8, alpha=0.9)
axes[0].fill_between(t_arr, 2, z2, where=(z2 >= 2), alpha=0.2, color='#c0392b')
axes[0].fill_between(t_arr, -2, z2, where=(z2 <= -2), alpha=0.2, color='#1e7e34')
for lvl, col, ls in [(2,'#c0392b','--'),(-2,'#1e7e34','--'),
                      (3,'#c0392b',':'),(-3,'#1e7e34',':')]:
    axes[0].axhline(lvl, linestyle=ls, color=col, lw=0.8)
axes[0].scatter(entradas_short, z2[entradas_short], marker='v',
                color='#c0392b', zorder=5, s=50, label='Entrada SHORT')
axes[0].scatter(entradas_long,  z2[entradas_long],  marker='^',
                color='#1e7e34', zorder=5, s=50, label='Entrada LONG')
axes[0].set_ylabel('Z-Score Kalman'); axes[0].set_ylim(-5, 5)
axes[0].legend(fontsize=9)
axes[0].text(T-1, 2.1, '+2σ (SHORT)', color='#c0392b', fontsize=8, ha='right')
axes[0].text(T-1, -2.4, '-2σ (LONG)',  color='#1e7e34', fontsize=8, ha='right')

# beta dinâmico
axes[1].plot(t_arr, betas2, color='#8e44ad', lw=1.0, label='β(t) Kalman')
axes[1].plot(t_arr, beta_real2, color='black', lw=1.0, ls='--',
             alpha=0.5, label='β real (oculto no modelo)')
axes[1].set_ylabel('β dinâmico'); axes[1].legend(fontsize=9)
axes[1].set_xlabel('Pregão')

plt.tight_layout()
plt.show()

print(f'\nSinais gerados nos 504 pregões:')
print(f'  LONG entries  : {len(entradas_long)}')
print(f'  SHORT entries : {len(entradas_short)}')
print(f'  % do tempo ativo: {((z2 >= 2) | (z2 <= -2)).mean():.1%}')

## 9. Como rodar o modelo completo

```python
# 1. Coloque na pasta do script:
#    - dados_economatica_B3.csv
#    - economatica_B3_setores.csv

# 2. Configure os parâmetros no topo do arquivo:
ANO_INICIO    = 2013   # 2 anos antes do 1º ciclo de backtest
ANO_FIM       = 2025
KALMAN_DELTA  = 1e-5   # velocidade de adaptação
KALMAN_N_WARM = 50     # obs para warm-start
GERAR_GRAFICOS = True
RAPIDO         = False  # True = ~40% mais rápido

# 3. Execute:
python 05_modelo.py

# Saídas em output/:
#   pares_por_janela.csv    — parâmetros Kalman por par por janela
#   zscores_kalman.csv      — série de z-score (wide format)
#   pares_frequentes.csv    — pares ordenados por frequência
#   graficos/05_beta_dinamico_kalman.png  — evolução do β
#   graficos/06_zscore_kalman_top6.png    — z-scores top 6
```

### Novos campos nos CSVs (vs versão OLS)

| Campo | Descrição |
|---|---|
| `Alpha_Kal` | Alpha final da janela de formação |
| `Beta_Kal_Final` | Beta final (valor atual do hedge ratio) |
| `Beta_Kal_Media` | Beta médio ao longo da janela |
| `Beta_Kal_Std` | Desvio padrão do beta — mede instabilidade |
| `Z_Score_Atual` | Z-score no último pregão da janela |

## 10. Interpretação do Beta_Kal_Std

`Beta_Kal_Std` é o desvio padrão de β(t) ao longo da janela de formação. É uma métrica de **estabilidade do hedge ratio**:

- **Baixo std** → relação estável (ex: PETR3/PETR4, ON/PN mesma empresa)
- **Alto std** → relação instável, possível quebra estrutural

Pares com `Beta_Kal_Std` alto devem ser operados com mais cautela — o hedge ratio pode mudar significativamente durante o trading.

### Sinal de alerta

```python
# Pares suspeitos: beta muito instável
pares_df = pd.read_csv('output/pares_por_janela.csv')
alertas = pares_df[pares_df['Beta_Kal_Std'] > 0.15]
print(f'{len(alertas)} pares com beta instável (std > 0.15)')
```

## 11. Limitações conhecidas

| Limitação | Impacto | Mitigação |
|---|---|---|
| In-sample: E-G e Kalman na mesma janela | Superestima qualidade dos pares na formação | Walk-forward no backtest resolve |
| Bias de sobrevivência | Ativos que saem do universo não penalizam PnL | Monitorar `situacao_cvm == CANCELADA` |
| Z-scores potencialmente não-gaussianos | Threshold ±2 pode ser inadequado | Ver diagnóstico de skewness no backtest |
| Warm-start influencia primeiras inovações | ~50 z-scores iniciais menos confiáveis | Considerar `KALMAN_N_WARM = 100` |
| Delta fixo para todos os pares | Pares ON/PN precisam delta menor | Calibração por setor (próxima fase) |

> **Nota importante**: o backtest (`06_backtest.py`) elimina o problema in-sample ao separar formação e trading em períodos distintos (ciclos bienais). O `05_modelo.py` é para geração de sinais correntes — o z-score da última janela é o sinal operacional atual.